# Feature Engineering: Indirect Prompt Injection Detection

Este notebook construye las características para detectar ataques de prompt injection usando clasificadores ML clásicos.

**Objetivo:** Extraer características lingüísticas y embeddings semánticos del dataset analizado en el EDA.

**Estructura:**
1. Load Dataset & EDA Summary
2. Linguistic / Handcrafted Features
3. Semantic Embedding Features
4. Feature Combination & Final Feature Matrix
5. Feature Importance Analysis
6. Save Artifacts

In [3]:
import re
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---
## SECTION 1 — Load Dataset & EDA Summary

Cargamos el dataset de indirect prompt injection y resumimos los hallazgos clave del EDA.

In [4]:
def resolve_data_path() -> Path:
    candidates = [
        Path("../data/indirect_prompt_injection_bipia_gpt_train.csv"),
        Path("data/indirect_prompt_injection_bipia_gpt_train.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Dataset CSV not found. Expected one of: "
        + ", ".join(str(p) for p in candidates)
    )

DATA_PATH = resolve_data_path()
df = pd.read_csv(DATA_PATH)

print(f"Dataset path: {DATA_PATH}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset path: ../data/indirect_prompt_injection_bipia_gpt_train.csv
Shape: (70000, 4)
Columns: ['context', 'user_intent', 'label', 'source']


In [5]:
# Verificar distribución de clases
print("Class distribution:")
print(df["label"].value_counts().sort_index())
print(f"\nTotal samples: {len(df)}")
print(f"Benign (0): {(df['label'] == 0).sum()} ({(df['label'] == 0).mean()*100:.1f}%)")
print(f"Malicious (1): {(df['label'] == 1).sum()} ({(df['label'] == 1).mean()*100:.1f}%)")

Class distribution:
label
0    35000
1    35000
Name: count, dtype: int64

Total samples: 70000
Benign (0): 35000 (50.0%)
Malicious (1): 35000 (50.0%)


In [6]:
# Verificar valores nulos
print("Null values per column:")
print(df.isna().sum())
assert df.isna().sum().sum() == 0, "Dataset contains null values!"

Null values per column:
context        0
user_intent    0
label          0
source         0
dtype: int64


### Conclusiones del EDA

Del análisis exploratorio previo se identificaron los siguientes hallazgos relevantes para feature engineering:

1. **Balance de clases:** Dataset perfectamente balanceado (35,000 benignos / 35,000 maliciosos)

2. **Confounding variable:** `label` está alineado 1:1 con `source` (BIPIA → malicious, GPT-4o-mini → benign). Esto implica que el modelo podría aprender patrones de estilo/origen en lugar de semántica de inyección.

3. **Longitudes discriminativas:**
   - `context`: mediana 961 chars, p95 = 3,305 chars
   - `user_intent`: mediana 56 chars, p95 = 343 chars
   - Los ejemplos maliciosos tienden a tener contextos con instrucciones embebidas

4. **Tipos de contenido detectados:**
   - Tablas Markdown: ~28.9%
   - Code fences: ~19.7%
   - Formato email: ~11.3%
   - URLs: ~8.9% (más frecuente en maliciosos: 14.8% vs 3%)
   - SQL keywords: ~7.9% (más en maliciosos: 10.1% vs 5.6%)

5. **Patrones de inyección:**
   - Frases directas como "ignore previous" no aparecen con alta frecuencia
   - "do not" aparece más en maliciosos (10.7% vs 3.9%)
   - Las inyecciones son sutiles y embebidas en contenido externo

**Implicación para features:** Necesitamos características que capturen tanto patrones léxicos explícitos como estructura semántica subyacente.

In [7]:
# Preparar columna de texto para feature extraction
# Usamos 'context' ya que es donde se embeben las instrucciones maliciosas
df["text"] = df["context"].astype(str)
y = df["label"].values

print(f"Text column created from 'context'")
print(f"Label array shape: {y.shape}")

Text column created from 'context'
Label array shape: (70000,)


---
## SECTION 2 — Linguistic / Handcrafted Features

Extraemos 10 características lingüísticas diseñadas específicamente para detectar prompt injection.
Cada feature está justificada en términos de su relevancia para la detección.

### Feature 1: `char_count` — Total character count

**Justificación:** Los prompts de inyección indirecta frecuentemente contienen instrucciones adicionales embebidas en el contexto, lo que puede resultar en textos más largos. Según el EDA, hay variación significativa en longitudes entre clases.

In [8]:
df["char_count"] = df["text"].str.len()
print(f"char_count - min: {df['char_count'].min()}, max: {df['char_count'].max()}, mean: {df['char_count'].mean():.2f}")

char_count - min: 31, max: 26406, mean: 1289.14


### Feature 2: `word_count` — Total word count

**Justificación:** Complementa char_count con una medida a nivel de palabra. Las instrucciones de inyección requieren vocabulario específico que puede aumentar el conteo de palabras.

In [9]:
df["word_count"] = df["text"].str.split().str.len()
print(f"word_count - min: {df['word_count'].min()}, max: {df['word_count'].max()}, mean: {df['word_count'].mean():.2f}")

word_count - min: 1, max: 6462, mean: 196.79


### Feature 3: `avg_word_length` — Average word length

**Justificación:** Textos técnicos o con comandos específicos pueden tener palabras más largas (e.g., "instructions", "override", "disregard"). Esta métrica captura complejidad léxica.

In [10]:
df["avg_word_length"] = df["char_count"] / df["word_count"].replace(0, 1)
print(f"avg_word_length - min: {df['avg_word_length'].min():.2f}, max: {df['avg_word_length'].max():.2f}, mean: {df['avg_word_length'].mean():.2f}")

avg_word_length - min: 3.38, max: 1772.00, mean: 7.08


### Feature 4: `sentence_count` — Number of sentences

**Justificación:** Las inyecciones de prompt frecuentemente incluyen múltiples oraciones imperativas o instrucciones. Un conteo alto de oraciones puede indicar instrucciones embebidas.

In [11]:
def count_sentences(text: str) -> int:
    """Count sentences by splitting on . ! ?"""
    sentences = re.split(r'[.!?]+', str(text))
    # Filter out empty strings
    return len([s for s in sentences if s.strip()])

df["sentence_count"] = df["text"].apply(count_sentences)
print(f"sentence_count - min: {df['sentence_count'].min()}, max: {df['sentence_count'].max()}, mean: {df['sentence_count'].mean():.2f}")

sentence_count - min: 1, max: 154, mean: 12.75


### Feature 5: `uppercase_ratio` — Ratio of uppercase chars to total chars

**Justificación:** Los atacantes a veces usan MAYÚSCULAS para enfatizar comandos o llamar la atención del modelo (e.g., "IGNORE", "DO NOT", "MUST"). Un ratio alto puede ser indicativo.

In [12]:
def uppercase_ratio(text: str) -> float:
    """Calculate ratio of uppercase characters to total characters."""
    text = str(text)
    if len(text) == 0:
        return 0.0
    uppercase_count = sum(1 for c in text if c.isupper())
    return uppercase_count / len(text)

df["uppercase_ratio"] = df["text"].apply(uppercase_ratio)
print(f"uppercase_ratio - min: {df['uppercase_ratio'].min():.4f}, max: {df['uppercase_ratio'].max():.4f}, mean: {df['uppercase_ratio'].mean():.4f}")

uppercase_ratio - min: 0.0000, max: 0.8530, mean: 0.0459


### Feature 6: `special_char_ratio` — Ratio of special characters to total chars

**Justificación:** Caracteres especiales como `|`, `>`, `$`, `{`, `}` son comunes en código, delimitadores de inyección, y markdown tables. El EDA mostró alta presencia de code fences y tablas.

In [13]:
SPECIAL_CHARS = set('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')

def special_char_ratio(text: str) -> float:
    """Calculate ratio of special characters to total characters."""
    text = str(text)
    if len(text) == 0:
        return 0.0
    special_count = sum(1 for c in text if c in SPECIAL_CHARS)
    return special_count / len(text)

df["special_char_ratio"] = df["text"].apply(special_char_ratio)
print(f"special_char_ratio - min: {df['special_char_ratio'].min():.4f}, max: {df['special_char_ratio'].max():.4f}, mean: {df['special_char_ratio'].mean():.4f}")

special_char_ratio - min: 0.0000, max: 0.4326, mean: 0.0623


### Feature 7: `injection_keyword_count` — Count of known injection trigger phrases

**Justificación:** Esta es la feature más directamente relacionada con prompt injection. Basado en literatura (Ayub & Majumdar 2024; Rahman et al. 2024) y patrones conocidos de jailbreaking. Incluye frases que intentan anular instrucciones del sistema.

In [14]:
INJECTION_KEYWORDS = [
    "ignore previous",
    "disregard",
    "forget instructions",
    "you are now",
    "pretend you are",
    "act as",
    "jailbreak",
    "DAN",
    "bypass",
    "override",
    "system prompt",
    "new instructions",
    "your real task",
    # Additional keywords based on EDA findings
    "do not",
    "don't",
    "must",
    "always",
    "never",
    "developer",
    "follow these instructions",
    "follow my instructions",
]

def count_injection_keywords(text: str) -> int:
    """Count occurrences of known injection trigger phrases."""
    text_lower = str(text).lower()
    count = 0
    for keyword in INJECTION_KEYWORDS:
        count += len(re.findall(re.escape(keyword.lower()), text_lower))
    return count

df["injection_keyword_count"] = df["text"].apply(count_injection_keywords)
print(f"injection_keyword_count - min: {df['injection_keyword_count'].min()}, max: {df['injection_keyword_count'].max()}, mean: {df['injection_keyword_count'].mean():.2f}")
print(f"Texts with at least 1 keyword: {(df['injection_keyword_count'] > 0).sum()} ({(df['injection_keyword_count'] > 0).mean()*100:.1f}%)")

injection_keyword_count - min: 0, max: 29, mean: 0.39
Texts with at least 1 keyword: 16522 (23.6%)


### Feature 8: `question_mark_count` — Number of ? characters

**Justificación:** Las preguntas son una forma común de intentar extraer información o cambiar el comportamiento del modelo. Un alto número de signos de interrogación puede indicar intentos de manipulación.

In [15]:
df["question_mark_count"] = df["text"].str.count(r'\?')
print(f"question_mark_count - min: {df['question_mark_count'].min()}, max: {df['question_mark_count'].max()}, mean: {df['question_mark_count'].mean():.2f}")

question_mark_count - min: 0, max: 30, mean: 0.20


### Feature 9: `imperative_verb_ratio` — Presence of imperative starters

**Justificación:** Comandos imperativos son la esencia de las inyecciones de prompt. Frases como "tell me", "give me", "show me", "do not" son indicadores de instrucciones directas al modelo.

In [16]:
IMPERATIVE_PATTERNS = [
    r"\btell me\b",
    r"\bgive me\b",
    r"\bshow me\b",
    r"\bdo not\b",
    r"\bdon't\b",
    r"\bmust\b",
    r"\balways\b",
    r"\bnever\b",
    r"\bplease\b",
    r"\bensure\b",
    r"\bmake sure\b",
    r"\byou should\b",
    r"\byou must\b",
    r"\byou will\b",
]

def imperative_verb_ratio(text: str) -> float:
    """Calculate ratio of imperative patterns to word count."""
    text_lower = str(text).lower()
    word_count = len(text_lower.split())
    if word_count == 0:
        return 0.0
    
    imperative_count = 0
    for pattern in IMPERATIVE_PATTERNS:
        imperative_count += len(re.findall(pattern, text_lower))
    
    return imperative_count / word_count

df["imperative_verb_ratio"] = df["text"].apply(imperative_verb_ratio)
print(f"imperative_verb_ratio - min: {df['imperative_verb_ratio'].min():.4f}, max: {df['imperative_verb_ratio'].max():.4f}, mean: {df['imperative_verb_ratio'].mean():.4f}")

imperative_verb_ratio - min: 0.0000, max: 0.0714, mean: 0.0027


### Feature 10: `language_switch_flag` — Binary flag for non-ASCII characters

**Justificación:** El uso de caracteres no-ASCII puede indicar intentos de evasión multilingüe o unicode smuggling. Atacantes pueden usar caracteres de otros idiomas o símbolos unicode para bypassear filtros.

In [17]:
def has_non_ascii(text: str) -> int:
    """Check if text contains non-ASCII characters (potential multilingual evasion)."""
    try:
        str(text).encode('ascii')
        return 0
    except UnicodeEncodeError:
        return 1

df["language_switch_flag"] = df["text"].apply(has_non_ascii)
print(f"language_switch_flag distribution:")
print(df["language_switch_flag"].value_counts())
print(f"\nTexts with non-ASCII: {df['language_switch_flag'].sum()} ({df['language_switch_flag'].mean()*100:.1f}%)")

language_switch_flag distribution:
language_switch_flag
0    44596
1    25404
Name: count, dtype: int64

Texts with non-ASCII: 25404 (36.3%)


### Consolidar DataFrame de Features Lingüísticas

In [18]:
LINGUISTIC_FEATURES = [
    "char_count",
    "word_count",
    "avg_word_length",
    "sentence_count",
    "uppercase_ratio",
    "special_char_ratio",
    "injection_keyword_count",
    "question_mark_count",
    "imperative_verb_ratio",
    "language_switch_flag",
]

df_linguistic = df[LINGUISTIC_FEATURES].copy()

print(f"df_linguistic shape: {df_linguistic.shape}")
print(f"\nFeature statistics:")
df_linguistic.describe()

df_linguistic shape: (70000, 10)

Feature statistics:


,char_count,word_count,avg_word_length,sentence_count,uppercase_ratio,special_char_ratio,injection_keyword_count,question_mark_count,imperative_verb_ratio,language_switch_flag
count,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,1289.137086,196.794200,7.081777,12.751114,0.045949,0.062345,0.390143,0.200757,0.002738,0.362914
std,959.061421,148.355385,14.762704,10.452247,0.035957,0.043868,0.955287,1.030886,0.006095,0.480844
min,31.000000,1.000000,3.381963,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,587.750000,84.000000,6.017533,6.000000,0.023284,0.028756,0.000000,0.000000,0.000000,0.000000
50%,961.000000,149.000000,6.383648,10.000000,0.034862,0.046016,0.000000,0.000000,0.000000,0.000000
75%,1770.000000,276.000000,7.010667,17.000000,0.054633,0.089572,0.000000,0.000000,0.002222,1.000000
max,26406.000000,6462.000000,1772.000000,154.000000,0.853047,0.432591,29.000000,30.000000,0.071429,1.000000
